# Customer Support Ticket Router - train & evaluate on Colab
**Runtime → Change runtime type → T4 GPU**, then *Runtime → Run all*. Total time ≈ 30-45 min.

All numbers in the presentation come from the output of this notebook (`results/results.md`).

In [ ]:
!nvidia-smi -L
!lscpu | grep 'Model name'

In [ ]:
# 1) get the code  (replace with YOUR GitHub username)
GITHUB_USER = "iremaknceng"
!git clone https://github.com/{GITHUB_USER}/ticket-router.git
%cd ticket-router

In [ ]:
!pip install -q -r requirements.txt

## 2) Smoke test (≈2 min) - runs the full pipeline on 98 hand-written examples to catch errors early

In [ ]:
!python train.py --smoke && python evaluate.py --smoke --runs 100

## 3) Full training: tf-idf+LR, word2vec+FFNN, mBERT teacher, distilled student, INT8 quantization

In [ ]:
!python train.py

## 4) Evaluate all models on the same CLINC150 + appliance test set (accuracy, OOS recall, size, CPU latency)

In [ ]:
!python evaluate.py --runs 200

In [ ]:
print(open('results/results.md').read())
import json; print(json.dumps(json.load(open('results/hardware.json')), indent=2))
print(open('models/train_log.json').read())

## 5) Failure examples for the Conclusion slide

In [ ]:
import pandas as pd
err = pd.read_csv('results/errors_student_int8.csv')
print(len(err), 'errors of the INT8 student'); err.sample(min(15, len(err)), random_state=0)

## 6) Upload weights to the Hugging Face Hub (needs a free HF account + *write* token)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_REPO = "iremaknceng/ticket-router-weights"
!python scripts/push_weights.py --repo {HF_REPO}

## 7) Download the results folder → commit it to GitHub (small JSON/CSV/MD files)

In [ ]:
!zip -r results.zip results
from google.colab import files; files.download('results.zip')

## 8) Optional: try the app right here (public gradio.live link) - you can record the demo video from it

In [ ]:
!python app.py --share